In [2]:
import pandas as pd

In [3]:
df_raw = pd.read_csv("online_retail.csv")

In [4]:
display(df_raw.head(5))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
eu_countries = ["Germany", "France", "Netherlands", "Spain", "Belgium", "Switzerland", "Portugal","Italy"]

In [6]:
uk_df = df_raw[df_raw["Country"] == "United Kingdom"]

In [7]:
eu_df = df_raw[df_raw["Country"].isin(eu_countries)]

In [8]:
row_df = df_raw[~df_raw["Country"].isin(eu_countries + ["United Kingdom"])]

In [9]:
uk_df.to_csv("region_uk.csv", index = "False")
eu_df.to_csv("region_eu.csv", index = "False")
row_df.to_csv("region_others.csv", index = "False")

In [10]:
files = ["region_uk.csv", "region_eu.csv", "region_others.csv"]
region_names = ["Uk","EU","Others"]


In [11]:
#creating an empty list named dfs
dfs=[]
#starting a loop which forms a temporary df named 'temp' and saves file name
#with its region name, InvoiceDates as new object, and a new column called
#'Region' with values as UK, EU, Others
for f, region in zip(files, region_names):
  temp = pd.read_csv(f, parse_dates=["InvoiceDate"])
  temp["Region"] = region
  dfs.append(temp)
#now dfs[] list has three items :
#dfs = [ <UK Dataframe>, <EU Dataframe>, <Others Dataframe> ]

In [12]:
#this line takes your 3 separate regional tables sitting in a list and glues
#them into a single table, with fresh clean row numbers — this unified table
#is now your one "data warehouse" table that represents all regions combined.

unified = pd.concat(dfs, ignore_index=True)

In [13]:
#cleaning rows with no customer_id
unified = unified.dropna(subset = ["Customer ID"])

In [14]:
#cleaning rows with cancelled invoice
unified = unified[~unified["Invoice"].astype(str).str.startswith("C")]
#here, astype(str) changes Invoice datatype to string forcefully, and then
# str.startswith("C") checks all rows with values starting eith C
# ~ does the opposite, keeping values where it doen't start with C
# unified[ ... ] — using that True/False result to filter the table, keeping
# only rows where the result is True (i.e., keeping only non-cancelled invoices).

In [15]:
print("Rows before cleaning : ", sum(len(d) for d in dfs))
print("Rows after cleaning : ", len(unified))
print(unified.head())

Rows before cleaning :  1067371
Rows after cleaning :  805620
   Unnamed: 0 Invoice StockCode                          Description  \
0           0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS   
1           1  489434    79323P                   PINK CHERRY LIGHTS   
2           2  489434    79323W                  WHITE CHERRY LIGHTS   
3           3  489434     22041         RECORD FRAME 7" SINGLE SIZE    
4           4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX   

   Quantity         InvoiceDate  Price  Customer ID         Country Region  
0        12 2009-12-01 07:45:00   6.95      13085.0  United Kingdom     Uk  
1        12 2009-12-01 07:45:00   6.75      13085.0  United Kingdom     Uk  
2        12 2009-12-01 07:45:00   6.75      13085.0  United Kingdom     Uk  
3        48 2009-12-01 07:45:00   2.10      13085.0  United Kingdom     Uk  
4        24 2009-12-01 07:45:00   1.25      13085.0  United Kingdom     Uk  


In [16]:
import sqlite3

In [17]:
conn = sqlite3.connect("retail.db")

In [20]:

unified.to_sql("orders", conn, if_exists="replace", index =False)

805620

In [21]:
orders_only = unified.groupby(["Region", "Customer ID", "Invoice"], as_index = False)["InvoiceDate"].min()
conn = sqlite3.connect("retail.db")
orders_only.to_sql("orders_deduped", conn, if_exists="replace", index =False)

36975

In [22]:
conn = sqlite3.connect("retail.db")

query = """
SELECT Region,
       ROUND(AVG(days_gap), 1) AS avg_reorder_days_gap,
       COUNT(*) AS num_reorders
FROM(
    SELECT Region,
    "Customer ID", InvoiceDate,
    LAG(InvoiceDate) OVER (PARTITION BY "Customer ID" ORDER BY InvoiceDate) AS prev_invoice_date,
    JULIANDAY(InvoiceDate) - JULIANDAY(LAG(InvoiceDate) OVER (PARTITION BY "Customer ID" ORDER BY InvoiceDate)) AS days_gap
    FROM orders_deduped
)sub
WHERE days_gap IS NOT NULL
GROUP BY Region
ORDER BY avg_reorder_days_gap DESC

"""


In [23]:
result = pd.read_sql(query, conn)
print(result)

   Region  avg_reorder_days_gap  num_reorders
0      Uk                  52.4         28194
1      EU                  50.4          1833
2  Others                  34.7          1067


In [25]:
print("Raw file rows:", len(df_raw))
print("Duplicate rows in raw file:", df_raw.duplicated().sum())

Raw file rows: 1067371
Duplicate rows in raw file: 34335


In [23]:
print(df_raw["InvoiceDate"].min(), df_raw["InvoiceDate"].max())

2009-12-01 07:45:00 2011-12-09 12:50:00


In [27]:
df_raw = pd.read_csv("online_retail.csv")
print("Total rows:", len(df_raw))
print("Duplicate rows:", df_raw.duplicated().sum())
print("Columns:", df_raw.columns.tolist())
print("Date range:", df_raw["InvoiceDate"].min(), "-", df_raw["InvoiceDate"].max())
print(df_raw["Country"].value_counts().head(10))

Total rows: 1067371
Duplicate rows: 34335
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Date range: 2009-12-01 07:45:00 - 2011-12-09 12:50:00
Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64


## Prep nd Export dashboard data for visualization

In [28]:
#using the same 'conn' daatabase connection object

dashboard_data = pd.read_sql(""" SELECT Region, "Customer ID", Invoice, InvoiceDate,
                                julianday(InvoiceDate) - julianday(
            LAG(InvoiceDate) OVER (PARTITION BY "Customer ID" ORDER BY InvoiceDate)
        ) AS days_gap
        FROM orders_deduped
 """, conn)

dashboard_data.to_csv("dashboard_data.csv", index = False)
print("Exported : ", len(dashboard_data), "rows")

Exported :  36975 rows


In [29]:
summary = pd.read_sql("""
    SELECT Region, ROUND(AVG(days_gap),1) AS avg_reorder_gap_days, COUNT(*) AS num_reorders
    FROM (
        SELECT Region, InvoiceDate,
        julianday(InvoiceDate) - julianday(LAG(InvoiceDate) OVER (PARTITION BY "Customer ID" ORDER BY InvoiceDate)) AS days_gap
        FROM orders_deduped
    ) sub
    WHERE days_gap IS NOT NULL
    GROUP BY Region
""", conn)
summary.to_csv("dashboard_summary.csv", index=False)
print(summary)

   Region  avg_reorder_gap_days  num_reorders
0      EU                  50.4          1833
1  Others                  34.7          1067
2      Uk                  52.4         28194
